# RAID Multi-Genre Analysis v2: Cross-Model Robustness

**Question:** Is the memory curve shape invariant to the *measuring model*, not just the text producer?

**Design:**
- Same RAID corpus: 8 genres x 6 sources (human + 5 AI models), ~1400 essays
- Run with **4 measuring models**: Mistral-7B (base), Mistral-7B-Instruct (RL'd), Llama-3-8B, Qwen2-7B
- **Random text control**: measure curves on token-shuffled text to quantify instrument bias
- Compare: do all measuring models recover the same curve shape?
- Key claim: the curve reflects structure *in the text*, not an artifact of the measuring instrument
- Secondary: does RLHF change how the model *measures* context decay?

**Based on:** RAID_Multi_Genre_Analysis_v1 (Mistral-7B only)

In [ ]:
# ── Install & Import ──
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
import statsmodels.formula.api as smf
from pathlib import Path
import json, math, time, gc, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
# ── Config ──
import os

# ── Detect environment and set paths ──
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    # Mount Drive
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v2")

    # Check if data exists on Drive
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
        print(f"Found corpus on Drive: {DATA_DIR}")
    else:
        # Fallback: upload to local Colab filesystem
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            print("Corpus not found on Drive. Upload raid_corpus.jsonl:")
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
            print(f"Saved to {LOCAL_DATA}")
        DATA_DIR = LOCAL_DATA
        print(f"Using local corpus: {DATA_DIR}")

    # Results always go to Drive (persistent)
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v2")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")
print(f"Corpus exists: {(DATA_DIR / 'raid_corpus.jsonl').exists()}")

# ── Measuring models ──
# Each entry: (short_name, HuggingFace model ID)
# Includes both base and instruct versions of Mistral to test RLHF effect
MEASURING_MODELS = [
    ("mistral-7b",      "mistralai/Mistral-7B-v0.1"),
    ("mistral-7b-inst", "mistralai/Mistral-7B-Instruct-v0.2"),
    ("llama3-8b",       "meta-llama/Meta-Llama-3-8B"),
    ("qwen2-7b",        "Qwen/Qwen2-7B"),
]

USE_4BIT = True

# ── Window config (same as v1) ──
WINDOWS = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
BURN_IN = 128
MAX_SCORE_TOKENS = 64
BUFFER = 10
MIN_TOKENS = BURN_IN + MAX_SCORE_TOKENS + BUFFER  # 202

# ── Random control config ──
N_RANDOM_CONTROLS = 30  # number of random-text docs to generate per measuring model
RANDOM_SEED = 42

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']
COLORS_POP = {'human': '#3498db', 'ai': '#e74c3c', 'random': '#2ecc71'}

print(f"\nMeasuring models: {[m[0] for m in MEASURING_MODELS]}")
print(f"Windows: {WINDOWS}")
print(f"Min tokens: {MIN_TOKENS}")
print(f"Random controls: {N_RANDOM_CONTROLS} docs per model")

In [ ]:
# ── Load corpus ──
corpus_path = DATA_DIR / "raid_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus)} documents")
print(f"\nBy domain x population:")
for d in DOMAINS:
    nh = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'human')
    na = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'ai')
    print(f"  {d:<15} human={nh}, ai={na}")

print(f"\nAI models: {sorted(set(c['model'] for c in corpus if c['population'] == 'ai'))}")

## Random Text Control

Generate control documents by randomly shuffling tokens from real human texts. This destroys all coherence structure while preserving token frequency distribution. Any curve shape measured on these documents reflects **instrument bias**, not text structure.

We use token-level shuffling (not word or sentence) to completely eliminate any local or long-range dependencies.

In [ ]:
# ── Core functions ──

def compute_perplexity_on_region(lm, token_ids, target_start, target_end):
    """Compute perplexity on tokens in [target_start, target_end)."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return float('inf'), 0
    input_ids = torch.tensor([token_ids], device=lm.device)
    with torch.no_grad():
        outputs = lm(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    return math.exp(total_loss / count) if count > 0 else float('inf'), count


def compute_memory_curve(lm, token_ids, windows, burn_in, max_score_tokens):
    """Compute perplexity at each context window size."""
    result = {'ppl_by_W': {}, 'token_count': len(token_ids)}
    n_tokens = len(token_ids)
    if n_tokens <= burn_in:
        return result
    target_end = min(n_tokens, burn_in + max_score_tokens)
    for W in windows:
        context_start = max(0, burn_in - W)
        actual_context = burn_in - context_start
        if actual_context < 4:
            continue
        truncated = token_ids[context_start:target_end]
        ppl, _ = compute_perplexity_on_region(lm, truncated, actual_context, len(truncated))
        if not math.isinf(ppl):
            result['ppl_by_W'][W] = ppl
    return result


def compute_half_life(ppl_dict, percentile=0.5):
    """Context length where `percentile` fraction of total benefit is achieved."""
    if len(ppl_dict) < 2:
        return float('nan')
    items = sorted(ppl_dict.items())
    windows = np.array([x[0] for x in items])
    ppls = np.array([x[1] for x in items])
    total_benefit = ppls[0] - ppls[-1]
    if total_benefit <= 0:
        return float('nan')
    target_ppl = ppls[0] - percentile * total_benefit
    for i in range(len(ppls) - 1):
        if ppls[i] >= target_ppl >= ppls[i + 1]:
            frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i + 1])
            return windows[i] + frac * (windows[i + 1] - windows[i])
    return windows[-1]


def extract_metrics(doc, ppl_by_W, n_tokens):
    """Extract all summary metrics from a perplexity-by-window dict."""
    computed_W = sorted(ppl_by_W.keys())
    computed_ppls = np.array([ppl_by_W[w] for w in computed_W])
    delta_ppl = computed_ppls[0] - computed_ppls

    row = {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'model': doc['model'],
        'population': doc['population'],
        'token_count': n_tokens,
        'half_life': compute_half_life(ppl_by_W),
    }

    for W in WINDOWS:
        row[f'ppl_W{W}'] = ppl_by_W.get(W, np.nan)

    row['delta_max'] = delta_ppl[-1]
    row['auc_full'] = trapezoid(delta_ppl, computed_W)

    if len(computed_W) >= 3:
        slope, _, _, _, _ = stats.linregress(np.log(computed_W), delta_ppl)
        row['log_slope'] = slope

    early_W = [w for w in computed_W if w <= 32]
    late_W = [w for w in computed_W if w >= 32]
    if len(early_W) >= 2:
        ep = np.array([ppl_by_W[w] for w in early_W])
        row['auc_early'] = trapezoid(ep[0] - ep, early_W)
    if len(late_W) >= 2:
        lp = np.array([ppl_by_W[w] for w in late_W])
        row['auc_late'] = trapezoid(lp[0] - lp, late_W)
    if row.get('auc_early') and row.get('auc_full') and row['auc_full'] > 0:
        row['early_fraction'] = row['auc_early'] / row['auc_full']

    return row


print("Functions defined")

# ── Main loop: run each measuring model ──

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

all_results = {}  # measuring_model_name -> DataFrame
random_results = {}  # measuring_model_name -> DataFrame (shuffled tokens)
uniform_results = {}  # measuring_model_name -> DataFrame (uniform random tokens)

for meas_name, meas_hf_id in MEASURING_MODELS:
    print(f"\n{'='*70}")
    print(f"MEASURING MODEL: {meas_name} ({meas_hf_id})")
    print(f"{'='*70}")

    results_path = BASE_DIR / f"essay_level_results_{meas_name}.csv"
    random_path = BASE_DIR / f"random_control_results_{meas_name}.csv"
    uniform_path = BASE_DIR / f"uniform_control_results_{meas_name}.csv"

    # Check what we already have
    have_essays = results_path.exists()
    have_random = random_path.exists()
    have_uniform = uniform_path.exists()

    if have_essays:
        all_results[meas_name] = pd.read_csv(results_path)
        print(f"  Loaded existing essay results: {len(all_results[meas_name])} rows")

    if have_random:
        random_results[meas_name] = pd.read_csv(random_path)
        print(f"  Loaded existing shuffled-random results: {len(random_results[meas_name])} rows")

    if have_uniform:
        uniform_results[meas_name] = pd.read_csv(uniform_path)
        print(f"  Loaded existing uniform-random results: {len(uniform_results[meas_name])} rows")

    # Skip if we have everything
    if have_essays and have_random and have_uniform:
        continue

    # Need to load model for remaining work
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(meas_hf_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16)
        lm = AutoModelForCausalLM.from_pretrained(
            meas_hf_id, quantization_config=bnb_config, device_map="auto")
    else:
        lm = AutoModelForCausalLM.from_pretrained(
            meas_hf_id, torch_dtype=torch.float16, device_map="auto")
    lm.eval()
    print(f"  Model loaded in {time.time() - t0:.0f}s")

    # ── Run real essays (if needed) ──
    if not have_essays:
        results = []
        skipped = 0

        for doc in tqdm(corpus, desc=f"  {meas_name} (essays)"):
            token_ids = tok.encode(doc["text"], add_special_tokens=False)
            n_tokens = len(token_ids)

            if n_tokens < MIN_TOKENS:
                skipped += 1
                continue

            curve = compute_memory_curve(lm, token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
            if len(curve['ppl_by_W']) < 3:
                skipped += 1
                continue

            row = extract_metrics(doc, curve['ppl_by_W'], n_tokens)
            results.append(row)

        df_model = pd.DataFrame(results)
        df_model['measuring_model'] = meas_name
        all_results[meas_name] = df_model

        print(f"  Essays: {len(df_model)} processed ({skipped} skipped)")
        df_model.to_csv(results_path, index=False)
        print(f"  Saved to {results_path}")

    # ── Run shuffled-token controls (if needed) ──
    if not have_random:
        rng = np.random.RandomState(RANDOM_SEED)
        human_docs = [d for d in corpus if d['population'] == 'human']
        sample_docs = rng.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)

        rand_rows = []
        for i, doc in enumerate(tqdm(sample_docs, desc=f"  {meas_name} (shuffled)")):
            token_ids = tok.encode(doc["text"], add_special_tokens=False)
            n_tokens = len(token_ids)

            if n_tokens < MIN_TOKENS:
                continue

            shuffled = list(token_ids)
            rng.shuffle(shuffled)

            curve = compute_memory_curve(lm, shuffled, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
            if len(curve['ppl_by_W']) < 3:
                continue

            rand_doc = {
                'doc_id': f'shuffled_{i:03d}',
                'domain': 'shuffled',
                'model': 'shuffled',
                'population': 'shuffled',
            }
            row = extract_metrics(rand_doc, curve['ppl_by_W'], n_tokens)
            rand_rows.append(row)

        df_random = pd.DataFrame(rand_rows)
        df_random['measuring_model'] = meas_name
        random_results[meas_name] = df_random

        print(f"  Shuffled controls: {len(df_random)} processed")
        df_random.to_csv(random_path, index=False)
        print(f"  Saved to {random_path}")

    # ── Run uniform-random controls (if needed) ──
    if not have_uniform:
        rng_u = np.random.RandomState(RANDOM_SEED + 1)
        vocab_size = tok.vocab_size
        # Match lengths to the human docs used for shuffled controls
        human_docs = [d for d in corpus if d['population'] == 'human']
        rng_len = np.random.RandomState(RANDOM_SEED)
        sample_docs = rng_len.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)

        uni_rows = []
        for i, doc in enumerate(tqdm(sample_docs, desc=f"  {meas_name} (uniform)")):
            # Match length of corresponding human doc
            token_ids = tok.encode(doc["text"], add_special_tokens=False)
            n_tokens = len(token_ids)

            if n_tokens < MIN_TOKENS:
                continue

            # Uniform random tokens from full vocabulary
            uniform_ids = rng_u.randint(0, vocab_size, size=n_tokens).tolist()

            curve = compute_memory_curve(lm, uniform_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
            if len(curve['ppl_by_W']) < 3:
                continue

            uni_doc = {
                'doc_id': f'uniform_{i:03d}',
                'domain': 'uniform',
                'model': 'uniform',
                'population': 'uniform',
            }
            row = extract_metrics(uni_doc, curve['ppl_by_W'], n_tokens)
            uni_rows.append(row)

        df_uniform = pd.DataFrame(uni_rows)
        df_uniform['measuring_model'] = meas_name
        uniform_results[meas_name] = df_uniform

        print(f"  Uniform controls: {len(df_uniform)} processed")
        df_uniform.to_csv(uniform_path, index=False)
        print(f"  Saved to {uniform_path}")

    # Unload model to free GPU
    del lm, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  Model unloaded, GPU memory freed")

# Combine all results
df_all = pd.concat(all_results.values(), ignore_index=True)
df_random_all = pd.concat(random_results.values(), ignore_index=True) if random_results else pd.DataFrame()
df_uniform_all = pd.concat(uniform_results.values(), ignore_index=True) if uniform_results else pd.DataFrame()

df_all.to_csv(BASE_DIR / "essay_level_results_all_models.csv", index=False)
if len(df_random_all) > 0:
    df_random_all.to_csv(BASE_DIR / "random_control_results_all_models.csv", index=False)
if len(df_uniform_all) > 0:
    df_uniform_all.to_csv(BASE_DIR / "uniform_control_results_all_models.csv", index=False)

print(f"\nCombined results: {len(df_all)} essay rows + {len(df_random_all)} shuffled + {len(df_uniform_all)} uniform")
print(f"Across {df_all.measuring_model.nunique()} measuring models")
print(f"Saved to {BASE_DIR}/")

In [ ]:
# ── Main loop: run each measuring model ──

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

all_results = {}  # measuring_model_name -> DataFrame

for meas_name, meas_hf_id in MEASURING_MODELS:
    print(f"\n{'='*70}")
    print(f"MEASURING MODEL: {meas_name} ({meas_hf_id})")
    print(f"{'='*70}")

    # Check if results already exist (resume support)
    results_path = BASE_DIR / f"essay_level_results_{meas_name}.csv"
    if results_path.exists():
        print(f"  Found existing results at {results_path}, loading...")
        all_results[meas_name] = pd.read_csv(results_path)
        print(f"  Loaded {len(all_results[meas_name])} rows")
        continue

    # Load tokenizer
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(meas_hf_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Load model
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16)
        lm = AutoModelForCausalLM.from_pretrained(
            meas_hf_id, quantization_config=bnb_config, device_map="auto")
    else:
        lm = AutoModelForCausalLM.from_pretrained(
            meas_hf_id, torch_dtype=torch.float16, device_map="auto")
    lm.eval()
    print(f"  Model loaded in {time.time() - t0:.0f}s")

    # Run all essays
    results = []
    skipped = 0

    for doc in tqdm(corpus, desc=f"  {meas_name}"):
        token_ids = tok.encode(doc["text"], add_special_tokens=False)
        n_tokens = len(token_ids)

        if n_tokens < MIN_TOKENS:
            skipped += 1
            continue

        curve = compute_memory_curve(lm, token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if len(curve['ppl_by_W']) < 3:
            skipped += 1
            continue

        row = extract_metrics(doc, curve['ppl_by_W'], n_tokens)
        results.append(row)

    df_model = pd.DataFrame(results)
    df_model['measuring_model'] = meas_name
    all_results[meas_name] = df_model

    print(f"  Processed {len(df_model)} docs ({skipped} skipped)")
    print(f"  Human: {len(df_model[df_model.population == 'human'])}")
    print(f"  AI:    {len(df_model[df_model.population == 'ai'])}")

    # Save per-model results
    df_model.to_csv(results_path, index=False)
    print(f"  Saved to {results_path}")

    # Unload model to free GPU
    del lm, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  Model unloaded, GPU memory freed")

# Combine all results
df_all = pd.concat(all_results.values(), ignore_index=True)
df_all.to_csv(BASE_DIR / "essay_level_results_all_models.csv", index=False)
print(f"\nCombined results: {len(df_all)} rows across {df_all.measuring_model.nunique()} measuring models")
print(f"Saved to {BASE_DIR / 'essay_level_results_all_models.csv'}")

## Figure 1: Same Text, Different Measuring Models

Key test: for the **same essays**, do different measuring models recover the same normalized curve shape? Each panel = one genre, each line = one measuring model. If curves overlap, the shape is a property of the text, not the instrument.

In [ ]:
# ── Figure 1: Normalized curves by measuring model (human text only) ──
ppl_cols = [f'ppl_W{w}' for w in WINDOWS]
plot_W = WINDOWS

meas_names = sorted(df_all['measuring_model'].unique())
meas_colors = dict(zip(meas_names, plt.cm.Set1.colors[:len(meas_names)]))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, domain in enumerate(DOMAINS):
    ax = axes[idx // 4, idx % 4]

    for meas_name in meas_names:
        sub = df_all[(df_all.domain == domain)
                     & (df_all.population == 'human')
                     & (df_all.measuring_model == meas_name)]
        if len(sub) == 0:
            continue
        means = np.array([sub[c].mean() for c in ppl_cols])
        total_drop = means[0] - means[-1]
        if total_drop > 0:
            normalized = (means[0] - means) / total_drop
        else:
            normalized = np.zeros_like(means)
        ax.plot(plot_W, normalized, marker='o', linewidth=2,
                color=meas_colors[meas_name], markersize=4, label=meas_name)

    ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(domain, fontsize=12, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    if idx % 4 == 0:
        ax.set_ylabel('Fraction of Total Benefit')
    if idx >= 4:
        ax.set_xlabel('Context Window')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=7)

plt.suptitle('Normalized Curves by Measuring Model (Human Text Only)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_cross_model_human.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2: Half-Life and Early Fraction Across Measuring Models

Do the scalar summary metrics agree across measuring models? Each group of bars = one genre, each color = one measuring model.

In [ ]:
# ── Figure 2: Half-life & early fraction by measuring model (human only) ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

n_models = len(meas_names)
bar_width = 0.8 / n_models

for idx, (metric, label) in enumerate([
    ('half_life', 'Half-Life (tokens)'),
    ('early_fraction', 'Early Fraction'),
    ('ppl_W4', 'Baseline Perplexity (W4)')
]):
    ax = axes[idx]
    x = np.arange(len(DOMAINS))

    for mi, meas_name in enumerate(meas_names):
        sub = df_all[(df_all.population == 'human') & (df_all.measuring_model == meas_name)]
        means = [sub[sub.domain == d][metric].dropna().mean() for d in DOMAINS]
        sems = [stats.sem(sub[sub.domain == d][metric].dropna()) if len(sub[sub.domain == d][metric].dropna()) > 1 else 0 for d in DOMAINS]
        offset = (mi - (n_models - 1) / 2) * bar_width
        ax.bar(x + offset, means, bar_width, yerr=sems,
               label=meas_name, color=meas_colors[meas_name], alpha=0.8, capsize=2)

    ax.set_xticks(x)
    ax.set_xticklabels(DOMAINS, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_metrics_cross_model.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3: Human vs AI Invariance Holds Across Measuring Models

Replicate the v1 finding (human vs AI curves overlap) separately for each measuring model. If all three models show the same invariance, this is strong evidence.

In [ ]:
# ── Figure 3: Human vs AI by measuring model (one row per measuring model) ──
fig, axes = plt.subplots(len(meas_names), len(DOMAINS), figsize=(24, 4 * len(meas_names)))

for row_idx, meas_name in enumerate(meas_names):
    for col_idx, domain in enumerate(DOMAINS):
        ax = axes[row_idx, col_idx] if len(meas_names) > 1 else axes[col_idx]

        for pop, ls, lw in [('human', '-', 2.5), ('ai', '--', 1.5)]:
            sub = df_all[(df_all.domain == domain)
                         & (df_all.population == pop)
                         & (df_all.measuring_model == meas_name)]
            if len(sub) == 0:
                continue
            means = np.array([sub[c].mean() for c in ppl_cols])
            total_drop = means[0] - means[-1]
            if total_drop > 0:
                normalized = (means[0] - means) / total_drop
            else:
                normalized = np.zeros_like(means)
            ax.plot(plot_W, normalized, marker='o', linestyle=ls, linewidth=lw,
                    color=COLORS_POP[pop], markersize=3, label=pop)

        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
        ax.set_xscale('log', base=2)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=0.2)

        if row_idx == 0:
            ax.set_title(domain, fontsize=11, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(f'{meas_name}\nFraction of Benefit', fontsize=10)
        if row_idx == len(meas_names) - 1:
            ax.set_xlabel('Context Window', fontsize=9)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=7)

plt.suptitle('Human (solid) vs AI (dashed): Each Row = Different Measuring Model',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig3_human_vs_ai_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Tests: Cross-Model Agreement

In [ ]:
# ── Cross-model agreement on half-life (per-essay correlation) ──
# For essays measured by all models, correlate half-life across measuring models
print("="*70)
print("CROSS-MODEL AGREEMENT: Per-Essay Half-Life Correlations")
print("="*70)

# Find essays common to all measuring models
from itertools import combinations

for pop_label, pop_filter in [("Human only", "human"), ("AI only", "ai"), ("All", None)]:
    print(f"\n--- {pop_label} ---")

    for m1, m2 in combinations(meas_names, 2):
        df1 = df_all[df_all.measuring_model == m1][['doc_id', 'half_life']].rename(
            columns={'half_life': f'hl_{m1}'})
        df2 = df_all[df_all.measuring_model == m2][['doc_id', 'half_life']].rename(
            columns={'half_life': f'hl_{m2}'})
        merged = df1.merge(df2, on='doc_id')

        if pop_filter:
            pop_docs = set(df_all[df_all.population == pop_filter]['doc_id'])
            merged = merged[merged.doc_id.isin(pop_docs)]

        merged = merged.dropna()
        if len(merged) < 5:
            print(f"  {m1} vs {m2}: insufficient overlap")
            continue

        r, p = stats.pearsonr(merged[f'hl_{m1}'], merged[f'hl_{m2}'])
        rho, p_rho = stats.spearmanr(merged[f'hl_{m1}'], merged[f'hl_{m2}'])
        print(f"  {m1} vs {m2}: r={r:.3f} (p={p:.1e}), rho={rho:.3f} (p={p_rho:.1e}), n={len(merged)}")

In [ ]:
# ── Does the human vs AI invariance replicate across measuring models? ──
print("="*70)
print("HUMAN vs AI SHAPE INVARIANCE: Per Measuring Model")
print("="*70)

for meas_name in meas_names:
    print(f"\n--- Measuring model: {meas_name} ---")
    sub = df_all[df_all.measuring_model == meas_name]

    for metric in ['half_life', 'early_fraction']:
        if metric not in sub.columns:
            continue
        print(f"\n  {metric}:")
        print(f"  {'Domain':<15} {'Human':>10} {'AI':>10} {'d':>8} {'p':>10}")
        print(f"  {'-'*55}")
        for d in DOMAINS:
            h = sub[(sub.domain == d) & (sub.population == 'human')][metric].dropna()
            a = sub[(sub.domain == d) & (sub.population == 'ai')][metric].dropna()
            if len(h) < 3 or len(a) < 3:
                continue
            t, p = stats.ttest_ind(h, a)
            cohen_d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            print(f"  {d:<15} {h.mean():>10.2f} {a.mean():>10.2f} {cohen_d:>8.3f} {p:>9.4f} {sig}")

In [ ]:
# ── Summary ──
print("="*70)
print("SUMMARY: Cross-Model Robustness Analysis")
print("="*70)

print(f"\nDataset: RAID ({len(corpus)} docs)")
print(f"Domains: {DOMAINS}")
print(f"Measuring models: {[m[0] for m in MEASURING_MODELS]}")
print(f"Windows: {WINDOWS}")

print(f"\n1. CURVE SHAPE ACROSS MEASURING MODELS:")
print(f"   Mean half-life (human text only) by measuring model:")
for meas_name in meas_names:
    sub = df_all[(df_all.measuring_model == meas_name) & (df_all.population == 'human')]
    hl = sub['half_life'].dropna()
    print(f"   {meas_name:<20}: {hl.mean():.1f} +/- {hl.std():.1f} tokens")

print(f"\n2. RAW PERPLEXITY DIFFERS (expected — different models):")
for meas_name in meas_names:
    sub = df_all[(df_all.measuring_model == meas_name) & (df_all.population == 'human')]
    ppl = sub['ppl_W4'].dropna()
    print(f"   {meas_name:<20}: baseline ppl = {ppl.mean():.1f} +/- {ppl.std():.1f}")

print(f"\n3. HUMAN vs AI INVARIANCE:")
for meas_name in meas_names:
    sub = df_all[df_all.measuring_model == meas_name]
    h = sub[sub.population == 'human']['half_life'].dropna()
    a = sub[sub.population == 'ai']['half_life'].dropna()
    t, p = stats.ttest_ind(h, a)
    d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
    sig = "YES" if p < 0.05 else "NO"
    print(f"   {meas_name:<20}: human={h.mean():.1f}, ai={a.mean():.1f}, d={d:.3f}, p={p:.4f} -> diff? {sig}")

print(f"\n4. RANDOM CONTROL:")
for meas_name in [m[0] for m in MEASURING_MODELS]:
    sub = df_random_all[df_random_all.measuring_model == meas_name]
    if len(sub) == 0:
        continue
    hl = sub['half_life'].dropna()
    dm = sub['delta_max'].dropna()
    print(f"   {meas_name:<20}: half-life={hl.mean():.1f}, delta_max={dm.mean():.2f}")

print(f"\n5. BASE vs INSTRUCT (RLHF EFFECT ON MEASUREMENT):")
for mn in ['mistral-7b', 'mistral-7b-inst']:
    sub = df_all[(df_all.measuring_model == mn) & (df_all.population == 'human')]
    if len(sub) == 0:
        continue
    hl = sub['half_life'].dropna()
    ef = sub['early_fraction'].dropna()
    print(f"   {mn:<20}: half-life={hl.mean():.1f}, early_fraction={ef.mean():.4f}")

print(f"\nAll results saved to {BASE_DIR}/")

## Figure 5: Random Text Control

If the models impose their own curve shape regardless of input, then random (token-shuffled) text should show a similar normalized curve to real text. If the curve reflects text structure, random text should look fundamentally different — flat or linear.

In [ ]:
# ── Figure 5: Real text vs shuffled vs uniform random, per measuring model ──
fig, axes = plt.subplots(1, len(MEASURING_MODELS), figsize=(6 * len(MEASURING_MODELS), 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

COLORS_CTRL = {'human': '#3498db', 'ai': '#e74c3c', 'shuffled': '#2ecc71', 'uniform': '#9b59b6'}

for idx, (meas_name, _) in enumerate(MEASURING_MODELS):
    ax = axes[idx]

    # Human curve
    sub_h = df_all[(df_all.population == 'human') & (df_all.measuring_model == meas_name)]
    if len(sub_h) > 0:
        means_h = np.array([sub_h[c].mean() for c in ppl_cols])
        total_h = means_h[0] - means_h[-1]
        if total_h > 0:
            ax.plot(plot_W, (means_h[0] - means_h) / total_h,
                    'o-', color=COLORS_CTRL['human'], linewidth=2.5, markersize=5, label='Human')

    # AI curve
    sub_a = df_all[(df_all.population == 'ai') & (df_all.measuring_model == meas_name)]
    if len(sub_a) > 0:
        means_a = np.array([sub_a[c].mean() for c in ppl_cols])
        total_a = means_a[0] - means_a[-1]
        if total_a > 0:
            ax.plot(plot_W, (means_a[0] - means_a) / total_a,
                    'o--', color=COLORS_CTRL['ai'], linewidth=1.5, markersize=4, label='AI')

    # Shuffled curve
    if len(df_random_all) > 0:
        sub_r = df_random_all[df_random_all.measuring_model == meas_name]
        if len(sub_r) > 0:
            avail_cols = [c for c in ppl_cols if c in sub_r.columns]
            means_r = np.array([sub_r[c].mean() for c in avail_cols])
            total_r = means_r[0] - means_r[-1]
            if total_r > 0:
                ax.plot(plot_W[:len(means_r)], (means_r[0] - means_r) / total_r,
                        's:', color=COLORS_CTRL['shuffled'], linewidth=2, markersize=5, label='Shuffled')

    # Uniform random curve
    if len(df_uniform_all) > 0:
        sub_u = df_uniform_all[df_uniform_all.measuring_model == meas_name]
        if len(sub_u) > 0:
            avail_cols = [c for c in ppl_cols if c in sub_u.columns]
            means_u = np.array([sub_u[c].mean() for c in avail_cols])
            total_u = means_u[0] - means_u[-1]
            if total_u > 0:
                ax.plot(plot_W[:len(means_u)], (means_u[0] - means_u) / total_u,
                        'D:', color=COLORS_CTRL['uniform'], linewidth=2, markersize=5, label='Uniform random')
            elif abs(total_u) < 0.01:
                # Flat line — no benefit from context (expected!)
                ax.axhline(0, color=COLORS_CTRL['uniform'], linestyle=':', linewidth=2,
                          alpha=0.5, label='Uniform random (flat)')

    # Reference: perfect diagonal
    ax.plot([plot_W[0], plot_W[-1]], [0, 1], 'k--', alpha=0.2, linewidth=1, label='Linear ref.')

    ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('Context Window')
    if idx == 0:
        ax.set_ylabel('Fraction of Total Benefit')
    ax.set_title(meas_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

plt.suptitle('Normalized Curves: Human vs AI vs Shuffled vs Uniform Random',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig5_controls.png', dpi=150, bbox_inches='tight')
plt.show()

# Print control stats
print("\nControl Summary:")
print(f"{'Condition':<15} {'Half-life':>10} {'Early Frac':>12} {'Delta Max':>10} {'PPL W4':>10}")
print("-"*60)
for meas_name, _ in MEASURING_MODELS:
    sub_h = df_all[(df_all.population == 'human') & (df_all.measuring_model == meas_name)]
    if len(sub_h) > 0:
        print(f"{'Human':<15} {sub_h['half_life'].mean():>10.1f} {sub_h['early_fraction'].mean():>12.4f} "
              f"{sub_h['delta_max'].mean():>10.2f} {sub_h['ppl_W4'].mean():>10.1f}  [{meas_name}]")

    if len(df_random_all) > 0:
        sub_r = df_random_all[df_random_all.measuring_model == meas_name]
        if len(sub_r) > 0:
            print(f"{'Shuffled':<15} {sub_r['half_life'].mean():>10.1f} {sub_r.get('early_fraction', pd.Series([np.nan])).mean():>12.4f} "
                  f"{sub_r['delta_max'].mean():>10.2f} {sub_r['ppl_W4'].mean():>10.1f}  [{meas_name}]")

    if len(df_uniform_all) > 0:
        sub_u = df_uniform_all[df_uniform_all.measuring_model == meas_name]
        if len(sub_u) > 0:
            print(f"{'Uniform':<15} {sub_u['half_life'].mean():>10.1f} {sub_u.get('early_fraction', pd.Series([np.nan])).mean():>12.4f} "
                  f"{sub_u['delta_max'].mean():>10.2f} {sub_u['ppl_W4'].mean():>10.1f}  [{meas_name}]")
    print()

## Figure 6: Base vs Instruct (RLHF Effect on Measurement)

Does RLHF change how the model *reads* context? Compare Mistral-7B-base and Mistral-7B-Instruct measuring the same human text. Same architecture, same training data, only difference is instruction tuning + RLHF.

In [ ]:
# ── Figure 6: Mistral base vs instruct on human text ──
base_name = 'mistral-7b'
inst_name = 'mistral-7b-inst'

if base_name in all_results and inst_name in all_results:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    for idx, domain in enumerate(DOMAINS):
        ax = axes[idx // 4, idx % 4]

        for mname, ls, lw, color, label in [
            (base_name, '-', 2.5, '#2c3e50', 'Mistral base'),
            (inst_name, '--', 2.0, '#e67e22', 'Mistral instruct'),
        ]:
            sub = df_all[(df_all.domain == domain)
                         & (df_all.population == 'human')
                         & (df_all.measuring_model == mname)]
            if len(sub) == 0:
                continue
            means = np.array([sub[c].mean() for c in ppl_cols])
            total_drop = means[0] - means[-1]
            if total_drop > 0:
                normalized = (means[0] - means) / total_drop
            else:
                normalized = np.zeros_like(means)
            ax.plot(plot_W, normalized, marker='o', linestyle=ls, linewidth=lw,
                    color=color, markersize=4, label=label)

        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
        ax.set_title(domain, fontsize=12, fontweight='bold')
        ax.set_xscale('log', base=2)
        ax.set_ylim(-0.05, 1.05)
        if idx % 4 == 0:
            ax.set_ylabel('Fraction of Total Benefit')
        if idx >= 4:
            ax.set_xlabel('Context Window')
        ax.grid(True, alpha=0.2)
        ax.legend(fontsize=8)

    plt.suptitle('RLHF Effect: Base vs Instruct Measuring Same Human Text',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig6_base_vs_instruct.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Per-essay correlation between base and instruct
    df_base = df_all[(df_all.measuring_model == base_name) & (df_all.population == 'human')][
        ['doc_id', 'half_life', 'early_fraction']].rename(
        columns={'half_life': 'hl_base', 'early_fraction': 'ef_base'})
    df_inst = df_all[(df_all.measuring_model == inst_name) & (df_all.population == 'human')][
        ['doc_id', 'half_life', 'early_fraction']].rename(
        columns={'half_life': 'hl_inst', 'early_fraction': 'ef_inst'})
    merged = df_base.merge(df_inst, on='doc_id').dropna()

    print(f"\nBase vs Instruct per-essay agreement (human text, n={len(merged)}):")
    for metric, col_b, col_i in [('half_life', 'hl_base', 'hl_inst'),
                                   ('early_fraction', 'ef_base', 'ef_inst')]:
        r, p = stats.pearsonr(merged[col_b], merged[col_i])
        mean_b = merged[col_b].mean()
        mean_i = merged[col_i].mean()
        print(f"  {metric}: base={mean_b:.2f}, inst={mean_i:.2f}, r={r:.3f} (p={p:.1e})")
else:
    print("Need both mistral-7b and mistral-7b-inst results to compare.")

In [ ]:
# ── Summary ──
print("="*70)
print("SUMMARY: Cross-Model Robustness Analysis")
print("="*70)

print(f"\nDataset: RAID ({len(corpus)} docs)")
print(f"Domains: {DOMAINS}")
print(f"Measuring models: {[m[0] for m in MEASURING_MODELS]}")
print(f"Windows: {WINDOWS}")

print(f"\n1. CURVE SHAPE ACROSS MEASURING MODELS:")
print(f"   Mean half-life (human text only) by measuring model:")
for meas_name in meas_names:
    sub = df_all[(df_all.measuring_model == meas_name) & (df_all.population == 'human')]
    hl = sub['half_life'].dropna()
    print(f"   {meas_name:<15}: {hl.mean():.1f} +/- {hl.std():.1f} tokens")

print(f"\n2. RAW PERPLEXITY DIFFERS (expected — different models):")
for meas_name in meas_names:
    sub = df_all[(df_all.measuring_model == meas_name) & (df_all.population == 'human')]
    ppl = sub['ppl_W4'].dropna()
    print(f"   {meas_name:<15}: baseline ppl = {ppl.mean():.1f} +/- {ppl.std():.1f}")

print(f"\n3. HUMAN vs AI INVARIANCE:")
for meas_name in meas_names:
    sub = df_all[df_all.measuring_model == meas_name]
    h = sub[sub.population == 'human']['half_life'].dropna()
    a = sub[sub.population == 'ai']['half_life'].dropna()
    t, p = stats.ttest_ind(h, a)
    d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
    sig = "YES" if p < 0.05 else "NO"
    print(f"   {meas_name:<15}: human={h.mean():.1f}, ai={a.mean():.1f}, d={d:.3f}, p={p:.4f} -> diff? {sig}")

print(f"\nAll results saved to {BASE_DIR}/")